In [ ]:
from tqdm import tqdm
from sklearn import datasets
import tensorflow as tf
from tensorflow.keras import layers, losses, metrics
import tensorflow_hessian as tfh

print("TensorFlow version:", tf.__version__)

datasets.load_digits

digits = datasets.load_digits()
X = digits.data
X = X / 16.0# + tf.random.normal(X.shape, mean=0.0, stddev=0.01)
t = digits.target


class MyModel(tfh.models.Model):
    def __init__(self):
        super().__init__()
        self.dense1 = tfh.layers.Dense(75)
        self.leaky_relu = layers.LeakyReLU()
        self.dense2 = tfh.layers.Dense(10)
        self.softmax = layers.Activation('softmax')

    def call(self, x, training=False):
        x = layers.Flatten()(x)
        x = self.dense1(x)
        x = self.leaky_relu(x)
        x = self.dense2(x)
        x = self.softmax(x)
        return x

model = MyModel()

optimizer = tfh.optimizers.MomentumNewtonMethod(eta=0.01, mu=0.0, alpha=0.1)

BATCH_SIZE = len(X)
train_dataset = tf.data.Dataset.from_tensor_slices((X, t)).batch(BATCH_SIZE)

def step(X, t, training=True):

    if training:
        with tf.GradientTape() as tape1:
            with tf.GradientTape() as tape2:
                y = model(X, training=training)
                loss = losses.SparseCategoricalCrossentropy()(t, y)
            grads = tape2.gradient(loss, model.trainable_variables)
        hessians = tape1.jacobian(grads[0], model.trainable_variables)
        optimizer.apply_gradients(model.trainable_variables, grads, hessians)
    else:
        y = model(X, training=training)
        loss = losses.SparseCategoricalCrossentropy()(t, y)

    accuracy = metrics.SparseCategoricalAccuracy()(t, y)
    return loss, accuracy

for epoch in (pb := tqdm(range(100))):

    total_loss = 0.0
    total_accuracy = 0.0
    total_data = 0
    for X, t in train_dataset:
        loss, accuracy = step(X, t, training=True)

        total_loss += loss.numpy() * X.shape[0] 
        total_accuracy += accuracy.numpy() * X.shape[0]
        total_data += X.shape[0]
        pb.set_postfix({"loss": total_loss / total_data, "accuracy": total_accuracy / total_data})

2025-10-04 16:14:49.329408: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1759562089.339416  792066 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1759562089.342300  792066 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1759562089.351333  792066 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1759562089.351348  792066 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1759562089.351350  792066 computation_placer.cc:177] computation placer alr

TensorFlow version: 2.19.0


I0000 00:00:1759562091.592201  792066 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 21458 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:01:00.0, compute capability: 8.9
  0%|          | 0/100 [00:00<?, ?it/s]2025-10-04 16:14:53.056626: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 3037828500 exceeds 10% of free system memory.
2025-10-04 16:14:54.137759: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 3037828500 exceeds 10% of free system memory.
I0000 00:00:1759562096.268364  792066 cuda_solvers.cc:175] Creating GpuSolver handles for stream 0x20733b50
  1%|          | 1/100 [00:04<07:23,  4.48s/it, loss=45.4, accuracy=0.104]2025-10-04 16:14:57.351545: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 3037828500 exceeds 10% of free system memory.
2025-10-04 16:14:58.065454: W external/local_xla/xla/tsl/framework/cpu_allocator_imp

  5%|▌         | 5/100 [00:15<04:41,  2.97s/it, loss=34.4, accuracy=0.127]

100%|██████████| 100/100 [04:49<00:00,  2.90s/it, loss=3.15, accuracy=0.721]
